# Milestone 4 — M4C: diffusion solve

NGSolve FEM fluence `Φ` on the coarse tet mesh (Robin boundary).

Plan: [`plans/milestone_04/04_diffusion_plan.md`](../../plans/milestone_04/04_diffusion_plan.md). Previous: `04b`. Next: `04d`.

## Notes

- Use **`S_clean`** as the FEM source term (volume-normalized), not raw `E_scat_elem`.
- `extrapolation_length` is macroscopic Robin leakage, not a Fresnel geometric length.


In [ ]:
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

!pip install --quiet --no-cache-dir "{ROOT}[fem]" -c "{ROOT}/requirements.txt"

from gummybear.paths import display_path

print(f"ROOT={{display_path(ROOT)}}")



In [7]:
import numpy as np
from gummybear.geometry import inspect_stl
from gummybear.optics import (
    OpticalMaterialConfig,
    deposit_ray_source,
    generate_diffusion_mesh,
    solve_diffusion,
)
from gummybear_validation.helpers import assert_live_netgen_mesh, make_centroid_axis_ray
from gummybear_validation.plotting import plot_phi_on_nodes


In [8]:
inspection = inspect_stl(ROOT / "cad" / "proto_bear_head.stl")
mesh = inspection["mesh"]
diff_mesh = generate_diffusion_mesh(mesh, target_elements=2000)
assert_live_netgen_mesh(diff_mesh)

material = OpticalMaterialConfig(mu_scatter=0.2, mu_absorption=0.001)

# Point source on central tet (sanity).
center = diff_mesh.centroids.mean(axis=0)
source_idx = int(np.argmin(np.linalg.norm(diff_mesh.centroids - center, axis=1)))
S_test = np.zeros(diff_mesh.n_tets)
S_test[source_idx] = 1.0


In [ ]:


solved_point = solve_diffusion(
    diff_mesh,
    S_clean=S_test,
    D=material.diffusion_coefficient,
    mu_a=material.mu_a,
)
print("point source: Phi_nodes", solved_point.Phi_nodes.min(), solved_point.Phi_nodes.mean(), solved_point.Phi_nodes.max())
print("residual_norm:", solved_point.residual_norm)
plot_phi_on_nodes(diff_mesh.nodes, solved_point.Phi_nodes, title="Phi — point source test")


In [ ]:
ray, p0, p1 = make_centroid_axis_ray(diff_mesh, axis="x", intensity=1.0)
dep = deposit_ray_source(diff_mesh, ray, mu_s=material.mu_s, mu_a=material.mu_a)

solved_ray = solve_diffusion(
    diff_mesh,
    S_clean=dep.S_clean,
    D=material.diffusion_coefficient,
    mu_a=material.mu_a,
    extrapolation_length=1000.0,
)
print("axis-ray deposition: residual_norm", solved_ray.residual_norm)
print("robin_boundary_model:", solved_ray.robin_boundary_model)
plot_phi_on_nodes(diff_mesh.nodes, solved_ray.Phi_nodes, log_scale=False, title="Phi — ray-deposited S_clean")
